In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../results/results.csv", keep_default_na=False)
default = df[(df.ways == 4) & (df.line_size == 32)].copy()
default["name"] = default.workload + default.variant.map(lambda v: f" ({v})" if v else "")
print(default[["name", "accesses", "misses", "hit_rate", "total_cycles", "worst_window_cycles"]].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].barh(default.name, default.hit_rate)
axes[0].set_title("Hit rate")
axes[1].barh(default.name, default.worst_window_cycles)
axes[1].set_title("Worst 100-access window (cycles)")
axes[0].invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
def row(workload, variant="", **where):
    q = df[(df.workload == workload) & (df.variant == variant)]
    for key, value in where.items():
        q = q[q[key] == value]
    return q.iloc[0]

d = dict(ways=4, line_size=32)
ring, avg = row("ring", **d), row("movavg", **d)
aos, soa = row("structs", "aos", **d), row("structs", "soa", **d)
by_row, by_col = row("traversal", "row", **d), row("traversal", "col", **d)
one_way, two_way = row("conflict", ways=1), row("conflict", ways=2)
cold = row("coldstart", **d)
steady_window = 100 * 1  # 100 hits at 1 cycle each

print(f"ring       hit rate {ring.hit_rate:.1%}, {ring.misses} misses ({4096 // 32} cold lines)")
print(f"movavg     hit rate {avg.hit_rate:.2%}, {avg.misses} misses ({256 // 32} window lines)")
print(f"structs    aos {aos.misses} misses vs soa {soa.misses} ({aos.misses / soa.misses:.0f}x)")
print(f"traversal  col {by_col.misses} misses vs row {by_row.misses} ({by_col.misses / by_row.misses:.0f}x)")
print(f"conflict   1 way: {one_way.misses} of {one_way.accesses} miss; 2 ways: {two_way.misses}")
print(f"coldstart  worst window {cold.worst_window_cycles} vs {steady_window} when warm ({cold.worst_window_cycles / steady_window:.0f}x)")

In [ ]:
for name in df.workload.unique():
    sub = df[df.workload == name]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for variant in sub.variant.unique():
        s = sub[sub.variant == variant].sort_values("sweep_value")
        label = variant or name
        axes[0].plot(s.sweep_value, s.hit_rate, marker="o", label=label)
        axes[1].plot(s.sweep_value, s.worst_window_cycles, marker="o", label=label)
    param = sub.sweep_param.iloc[0]
    for ax, ylabel in zip(axes, ["Hit rate", "Worst 100-access window (cycles)"]):
        ax.set_xlabel(param)
        ax.set_ylabel(ylabel)
        ax.set_ylim(bottom=0)
        ax.legend()
    fig.suptitle(name)
    plt.tight_layout()
    plt.show()

In [ ]:
d = default.copy()
d["mean_cost"] = d.cost_mean
d["worst_window_per_access"] = d.worst_window_cycles / d.window
d["worst / mean"] = d.worst_window_per_access / d.mean_cost
print(d[["name", "mean_cost", "worst_window_per_access", "worst / mean"]].round(2).to_string(index=False))